In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the paths to your files in Google Drive
# Please ensure these paths are correct. You might need to adjust them based on your file location.
# For example, if your files are directly in 'My Drive', the path would be '/content/drive/MyDrive/filename.csv'
file_path_temp = '/content/drive/MyDrive/co_m_temp_1980_2025.csv' # Adjust this path if needed
file_path_swe = '/content/drive/MyDrive/yearly_swe_average.csv' # Adjust this path if needed

# Check if files exist before trying to load them
if not os.path.exists(file_path_temp):
    print(f"Error: File not found at {file_path_temp}")
elif not os.path.exists(file_path_swe):
    print(f"Error: File not found at {file_path_swe}")
else:
    # Load the datasets
    try:
        df_temp = pd.read_csv(file_path_temp)
        df_swe = pd.read_csv(file_path_swe)

        print("Datasets loaded successfully.")
        print("co_m_temp_1980_2025 head:")
        print(df_temp.head())
        print("\nyearly_swe_average head:")
        print(df_swe.head())

        # Merge the datasets on the 'year' column
        # Assuming 'year' column exists in both dataframes
        merged_df = pd.merge(df_temp, df_swe, on='year', how='inner')

        print("\nMerged DataFrame head:")
        print(merged_df.head())

        # Ensure the columns for correlation exist and are numeric
        if 'swe_average' in merged_df.columns and 'co_avg_temp' in merged_df.columns:
            # Convert columns to numeric, coercing errors will turn non-numeric into NaN
            merged_df['swe_average'] = pd.to_numeric(merged_df['swe_average'], errors='coerce')
            merged_df['co_avg_temp'] = pd.to_numeric(merged_df['co_avg_temp'], errors='coerce')

            # Drop rows with NaN values in the relevant columns for correlation calculation
            correlation_df = merged_df.dropna(subset=['swe_average', 'co_avg_temp'])

            if not correlation_df.empty:
                # Calculate the Pearson correlation coefficient
                pearson_corr = correlation_df['swe_average'].corr(correlation_df['co_avg_temp'], method='pearson')
                print(f"\nPearson correlation coefficient between 'swe_average' and 'co_avg_temp': {pearson_corr}")
            else:
                print("No valid data points left after dropping NaNs for correlation calculation.")
        else:
            print("Error: One or both of the required columns ('swe_average', 'co_avg_temp') not found in the merged dataset.")

    except Exception as e:
        print(f"An error occurred during data loading or processing: {e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datasets loaded successfully.
co_m_temp_1980_2025 head:
   year  co_avg_temp
0  1980         7.54
1  1981         8.27
2  1982         6.55
3  1983         6.71
4  1984         6.36

yearly_swe_average head:
   year  swe_average
0  1980     3.879167
1  1981     2.787755
2  1982     7.146000
3  1983     7.501961
4  1984     8.658000

Merged DataFrame head:
   year  co_avg_temp  swe_average
0  1980         7.54     3.879167
1  1981         8.27     2.787755
2  1982         6.55     7.146000
3  1983         6.71     7.501961
4  1984         6.36     8.658000

Pearson correlation coefficient between 'swe_average' and 'co_avg_temp': -0.6370757418049056


In [ ]:
import altair as alt

# Create the interactive scatter plot
chart = alt.Chart(correlation_df).mark_point().encode(
    x=alt.X('co_avg_temp', title='Colorado Average Temperature'),
    y=alt.Y('swe_average', title='Yearly SWE Average'),
    tooltip=['year', 'co_avg_temp', 'swe_average']
).properties(
    title='Yearly SWE Average vs. Colorado Average Temperature'
).interactive()

# Define the path to save the HTML file in Google Drive
output_path = '/content/drive/MyDrive/yearly_swe_temp_correlation.html'

# Save the chart as an HTML file
chart.save(output_path)

print(f"Interactive scatter plot saved to {output_path}")


Interactive scatter plot saved to /content/drive/MyDrive/yearly_swe_temp_correlation.html


In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from google.colab import files
import os

# Filter the data for years between 1980 and 2025 (inclusive)
filtered_df = correlation_df[(correlation_df['year'] >= 1980) & (correlation_df['year'] <= 2025)]

# Create subplots: 1 row, 2 columns
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Linear Regression of Colorado Average Temperature (1980-2025)",
                                    "Linear Regression of Yearly SWE Average (1980-2025)"))

# --- Plot for co_avg_temp ---
# Create a linear regression plot for co_avg_temp over the years using plotly.express
fig_temp_px = px.scatter(
    filtered_df,
    x='year',
    y='co_avg_temp',
    trendline='ols',
    labels={'co_avg_temp': 'Colorado Average Temperature (°C)'},
    hover_data={'year':True, 'co_avg_temp':':.2f'} # Add hover data for year and temperature
)

# Add traces from the px.scatter plot to the first subplot
for trace in fig_temp_px.data:
    fig.add_trace(trace, row=1, col=1)

# Update x and y axis titles for the first subplot manually
fig.update_xaxes(title_text='Year', row=1, col=1)
fig.update_yaxes(title_text='Colorado Average Temperature (°C)', row=1, col=1)

# --- Plot for swe_average ---
# Create a linear regression plot for swe_average over the years using plotly.express
fig_swe_px = px.scatter(
    filtered_df,
    x='year',
    y='swe_average',
    trendline='ols',
    labels={'swe_average': 'Yearly SWE Average'},
    hover_data={'year':True, 'swe_average':':.2f'} # Add hover data for year and SWE
)

# Add traces from the px.scatter plot to the second subplot
for trace in fig_swe_px.data:
    fig.add_trace(trace, row=1, col=2)

# Update x and y axis titles for the second subplot manually
fig.update_xaxes(title_text='Year', row=1, col=2)
fig.update_yaxes(title_text='Yearly SWE Average (Inches)', row=1, col=2)

# Update layout for the combined figure to include hover functionality
fig.update_layout(
    title_text='Snow Water Equivelency VS Colorado Average Temperature',
    height=500,
    showlegend=False,
    hovermode='x unified' # Enable unified hover mode with a vertical line
)

# Add Pearson correlation coefficient as an annotation below the plots
fig.add_annotation(
    xref='paper', yref='paper',
    x=0.5, y=-0.2, # Position below the plots
    text=f"Pearson correlation coefficient: {pearson_corr:.3f}",
    showarrow=False,
    font=dict(size=16)
)

# Define the output path for the combined HTML file
combined_output_path = '/content/combined_regression_plots.html'
fig.write_html(combined_output_path, auto_open=False)
print(f"Combined interactive plot saved to {combined_output_path}")

# Automatically download the combined file
if os.path.exists(combined_output_path):
    files.download(combined_output_path)
    print(f"Downloading {os.path.basename(combined_output_path)}...")
else:
    print(f"Error: {combined_output_path} not found for download.")


Combined interactive plot saved to /content/combined_regression_plots.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — works in Colab and headless environments
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from scipy.stats import linregress, pearsonr

# ── Google Colab-specific imports ──────────────────────────────────────────────
try:
    from google.colab import files, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Mount Google Drive ─────────────────────────────────────────────────────────
if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    file_path_temp = '/content/drive/MyDrive/co_m_temp_1980_2025.csv'
    file_path_swe  = '/content/drive/MyDrive/yearly_swe_average.csv'
else:
    file_path_temp = 'co_m_temp_1980_2025.csv'
    file_path_swe  = 'yearly_swe_average.csv'

OUTPUT_PATH = '/content/swe_temp_regression.png' if IN_COLAB else 'swe_temp_regression.png'

# ── Load & merge data ──────────────────────────────────────────────────────────
try:
    df_temp = pd.read_csv(file_path_temp)
    df_swe  = pd.read_csv(file_path_swe)
except FileNotFoundError as e:
    raise SystemExit(f"File not found: {e}. Check your Google Drive paths.")

merged_df = pd.merge(df_temp, df_swe, on='year', how='inner')
filtered_df = merged_df[(merged_df['year'] >= 1980) & (merged_df['year'] <= 2025)].copy()

required = {'swe_average', 'co_avg_temp', 'year'}
missing  = required - set(filtered_df.columns)
if missing:
    raise SystemExit(f"Missing columns in merged data: {missing}")

filtered_df['swe_average'] = pd.to_numeric(filtered_df['swe_average'], errors='coerce')
filtered_df['co_avg_temp'] = pd.to_numeric(filtered_df['co_avg_temp'], errors='coerce')
df = filtered_df.dropna(subset=['swe_average', 'co_avg_temp']).sort_values('year').reset_index(drop=True)

if df.empty:
    raise SystemExit("No valid data points after dropping NaNs.")

# ── Regression statistics ──────────────────────────────────────────────────────
slope, intercept, r_value, p_value, std_err = linregress(df['co_avg_temp'], df['swe_average'])
pearson_r, _ = pearsonr(df['co_avg_temp'], df['swe_average'])
r_squared    = r_value ** 2
n            = len(df)
p_display    = f"{p_value:.2e}" if p_value < 0.001 else f"{p_value:.4f}"

print("── Regression Results ────────────────────────────────")
print(f"  Pearson r  : {pearson_r:.6f}")
print(f"  R²         : {r_squared:.6f}")
print(f"  P-value    : {p_value:.6e}")
print(f"  Slope      : {slope:.4f}")
print(f"  Intercept  : {intercept:.4f}")
print(f"  Std Error  : {std_err:.6f}")
print(f"  N          : {n}")
print("──────────────────────────────────────────────────────")

# ── Trendline data ─────────────────────────────────────────────────────────────
x_line = np.linspace(df['co_avg_temp'].min(), df['co_avg_temp'].max(), 200)
y_line = slope * x_line + intercept

# Color-map points by year for extra information density
years      = df['year'].values
norm_years = (years - years.min()) / (years.max() - years.min())
cmap       = plt.cm.viridis

# ── Figure layout ──────────────────────────────────────────────────────────────
# Main scatter + a narrow bottom strip for the stat cards
fig = plt.figure(figsize=(10, 8), dpi=150, facecolor='#f5f7fa')
gs  = gridspec.GridSpec(
    2, 1,
    height_ratios=[5, 1],
    hspace=0.35,
    left=0.10, right=0.95, top=0.88, bottom=0.08
)

# ── Top panel: scatter + regression ───────────────────────────────────────────
ax = fig.add_subplot(gs[0])
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#cccccc')

# Scatter points, colored by year
sc = ax.scatter(
    df['co_avg_temp'], df['swe_average'],
    c=norm_years, cmap=cmap,
    s=70, zorder=3, edgecolors='white', linewidths=0.6, alpha=0.92
)

# Colorbar for year
cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label('Year', fontsize=9, labelpad=6)
tick_positions = [0, 0.25, 0.5, 0.75, 1.0]
cbar.set_ticks(tick_positions)
cbar.set_ticklabels([
    str(int(years.min() + t * (years.max() - years.min())))
    for t in tick_positions
], fontsize=8)

# Regression line
ax.plot(x_line, y_line, color='crimson', linewidth=2,
        linestyle='--', zorder=4, label='OLS Trend')

# Confidence band (95%)
x_mean   = df['co_avg_temp'].mean()
se_band  = std_err * np.sqrt(
    1/n + (x_line - x_mean)**2 / ((df['co_avg_temp'] - x_mean)**2).sum()
)
t_crit   = 2.0  # ~95% CI approximation
ax.fill_between(
    x_line,
    y_line - t_crit * se_band,
    y_line + t_crit * se_band,
    color='crimson', alpha=0.10, zorder=2, label='95% Confidence Band'
)

# Annotate each point with its year (small, offset)
for _, row in df.iterrows():
    ax.annotate(
        str(int(row['year'])),
        xy=(row['co_avg_temp'], row['swe_average']),
        xytext=(3, 4), textcoords='offset points',
        fontsize=5.5, color='#444444', alpha=0.85
    )

ax.set_xlabel('Colorado Average Temperature (°C)', fontsize=11, labelpad=8)
ax.set_ylabel('Yearly SWE Average (inches)',        fontsize=11, labelpad=8)
ax.tick_params(axis='both', labelsize=9, colors='#444')
ax.grid(True, linestyle='--', linewidth=0.5, color='#e0e0e0', zorder=0)
ax.legend(fontsize=9, framealpha=0.9, loc='upper right')

# Inline stats text box (upper left)
stats_lines = (
    f"y = {slope:.3f}x + {intercept:.3f}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"R² = {r_squared:.3f}\n"
    f"p-value = {p_display}\n"
    f"Std Error = {std_err:.4f}\n"
    f"n = {n}"
)
ax.text(
    0.02, 0.97, stats_lines,
    transform=ax.transAxes,
    fontsize=8.5, verticalalignment='top',
    fontfamily='monospace',
    bbox=dict(
        boxstyle='round,pad=0.5',
        facecolor='white', edgecolor='#aaaaaa',
        alpha=0.90, linewidth=0.8
    )
)

# ── Bottom panel: stat cards ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_axis_off()

stats = [
    ("Pearson r",  f"{pearson_r:.3f}"),
    ("R²",         f"{r_squared:.3f}"),
    ("p-value",    p_display),
    ("Slope",      f"{slope:.3f}"),
    ("Intercept",  f"{intercept:.3f}"),
    ("Std Error",  f"{std_err:.4f}"),
    ("n",          str(n)),
]

n_cards  = len(stats)
card_w   = 1.0 / n_cards
card_pad = 0.008

for i, (label, value) in enumerate(stats):
    x0 = i * card_w + card_pad
    # Card background
    fancy = FancyBboxPatch(
        (x0, 0.05), card_w - 2 * card_pad, 0.88,
        boxstyle="round,pad=0.02",
        transform=ax2.transAxes,
        facecolor='white', edgecolor='#cccccc',
        linewidth=0.8, clip_on=False
    )
    ax2.add_patch(fancy)

    cx = x0 + (card_w - 2 * card_pad) / 2 + card_pad

    # Value (large, bold)
    ax2.text(
        cx, 0.68, value,
        transform=ax2.transAxes,
        ha='center', va='center',
        fontsize=10, fontweight='bold', color='#1a1a2e'
    )
    # Label (small, grey)
    ax2.text(
        cx, 0.22, label,
        transform=ax2.transAxes,
        ha='center', va='center',
        fontsize=7.5, color='#888888',
        fontfamily='sans-serif'
    )

# ── Title ─────────────────────────────────────────────────────────────────────
fig.suptitle(
    'Yearly SWE Average vs. Colorado Average Temperature  (1980–2025)',
    fontsize=13, fontweight='bold', color='#1a1a2e', y=0.95
)

# ── Save ───────────────────────────────────────────────────────────────────────
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print(f"\nPNG saved to: {OUTPUT_PATH}")

# ── Auto-download in Colab ─────────────────────────────────────────────────────
if IN_COLAB and os.path.exists(OUTPUT_PATH):
    files.download(OUTPUT_PATH)
    print("Downloading file...")

Mounted at /content/drive
── Regression Results ────────────────────────────────
  Pearson r  : -0.637076
  R²         : 0.405866
  P-value    : 1.932497e-06
  Slope      : -1.5681
  Intercept  : 17.3797
  Std Error  : 0.286019
  N          : 46
──────────────────────────────────────────────────────

PNG saved to: /content/swe_temp_regression.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from scipy.stats import linregress, pearsonr

# ── Google Colab-specific imports ──────────────────────────────────────────────
try:
    from google.colab import files, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Mount Google Drive ─────────────────────────────────────────────────────────
if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    file_path_temp = '/content/drive/MyDrive/co_m_temp_1980_2025.csv'
    file_path_swe  = '/content/drive/MyDrive/yearly_swe_average.csv'
else:
    file_path_temp = 'co_m_temp_1980_2025.csv'
    file_path_swe  = 'yearly_swe_average.csv'

OUTPUT_PATH = '/content/swe_temp_regression.png' if IN_COLAB else 'swe_temp_regression.png'

# ── Load & merge data ──────────────────────────────────────────────────────────
try:
    df_temp = pd.read_csv(file_path_temp)
    df_swe  = pd.read_csv(file_path_swe)
except FileNotFoundError as e:
    raise SystemExit(f"File not found: {e}. Check your Google Drive paths.")

merged_df   = pd.merge(df_temp, df_swe, on='year', how='inner')
filtered_df = merged_df[(merged_df['year'] >= 1980) & (merged_df['year'] <= 2025)].copy()

required = {'swe_average', 'co_avg_temp', 'year'}
missing  = required - set(filtered_df.columns)
if missing:
    raise SystemExit(f"Missing columns in merged data: {missing}")

filtered_df['swe_average'] = pd.to_numeric(filtered_df['swe_average'], errors='coerce')
filtered_df['co_avg_temp'] = pd.to_numeric(filtered_df['co_avg_temp'], errors='coerce')
df = filtered_df.dropna(subset=['swe_average', 'co_avg_temp']).sort_values('year').reset_index(drop=True)

if df.empty:
    raise SystemExit("No valid data points after dropping NaNs.")

# ── Regression statistics ──────────────────────────────────────────────────────
slope, intercept, r_value, p_value, std_err = linregress(df['co_avg_temp'], df['swe_average'])
pearson_r, _  = pearsonr(df['co_avg_temp'], df['swe_average'])
r_squared     = r_value ** 2
n             = len(df)
p_display     = f"{p_value:.2e}" if p_value < 0.001 else f"{p_value:.4f}"

print("── Regression Results ────────────────────────────────")
print(f"  Pearson r  : {pearson_r:.6f}")
print(f"  R²         : {r_squared:.6f}")
print(f"  P-value    : {p_value:.6e}")
print(f"  Slope      : {slope:.4f}")
print(f"  Intercept  : {intercept:.4f}")
print(f"  Std Error  : {std_err:.6f}")
print(f"  N          : {n}")
print("──────────────────────────────────────────────────────")

# ── Trendline ──────────────────────────────────────────────────────────────────
x_line = np.linspace(df['co_avg_temp'].min(), df['co_avg_temp'].max(), 200)
y_line = slope * x_line + intercept

# ── Figure layout ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(10, 8), dpi=150, facecolor='#f5f7fa')
gs  = gridspec.GridSpec(
    2, 1,
    height_ratios=[5, 1],
    hspace=0.35,
    left=0.10, right=0.95, top=0.88, bottom=0.08
)

# ── Top panel: scatter + regression ───────────────────────────────────────────
ax = fig.add_subplot(gs[0])
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#cccccc')

# Uniform color scatter — no colormap, no legend
ax.scatter(
    df['co_avg_temp'], df['swe_average'],
    color='steelblue', s=70,
    zorder=3, edgecolors='white', linewidths=0.6, alpha=0.90
)

# Regression line — no label, so no legend entry
ax.plot(x_line, y_line, color='crimson', linewidth=2, linestyle='--', zorder=4)

# Year label on each point
for _, row in df.iterrows():
    ax.annotate(
        str(int(row['year'])),
        xy=(row['co_avg_temp'], row['swe_average']),
        xytext=(3, 4), textcoords='offset points',
        fontsize=5.5, color='#444444', alpha=0.85
    )

ax.set_xlabel('Colorado Average Temperature (°C)', fontsize=11, labelpad=8)
ax.set_ylabel('Yearly SWE Average (inches)',        fontsize=11, labelpad=8)
ax.tick_params(axis='both', labelsize=9, colors='#444')
ax.grid(True, linestyle='--', linewidth=0.5, color='#e0e0e0', zorder=0)

# Remove legend if matplotlib auto-created one
if ax.get_legend():
    ax.get_legend().remove()

# Stats text box
stats_lines = (
    f"y = {slope:.3f}x + {intercept:.3f}\n"
)
ax.text(
    0.98, 0.97, stats_lines,
    transform=ax.transAxes,
    fontsize=8.5, verticalalignment='top', horizontalalignment='right',
    fontfamily='monospace',
    bbox=dict(
        boxstyle='round,pad=0.5',
        facecolor='white', edgecolor='#aaaaaa',
        alpha=0.90, linewidth=0.8
    )
)

# ── Bottom panel: stat cards ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_axis_off()

stats = [
    ("Pearson r",  f"{pearson_r:.3f}"),
    ("R²",         f"{r_squared:.3f}"),
    ("p-value",    p_display),
    ("Slope",      f"{slope:.3f}"),
    ("Intercept",  f"{intercept:.3f}"),
    ("n",          str(n)),
]

n_cards  = len(stats)
card_w   = 1.0 / n_cards
card_pad = 0.008

for i, (label, value) in enumerate(stats):
    x0 = i * card_w + card_pad
    fancy = FancyBboxPatch(
        (x0, 0.05), card_w - 2 * card_pad, 0.88,
        boxstyle="round,pad=0.02",
        transform=ax2.transAxes,
        facecolor='white', edgecolor='#cccccc',
        linewidth=0.8, clip_on=False
    )
    ax2.add_patch(fancy)

    cx = x0 + (card_w - 2 * card_pad) / 2 + card_pad

    ax2.text(
        cx, 0.68, value,
        transform=ax2.transAxes,
        ha='center', va='center',
        fontsize=10, fontweight='bold', color='#1a1a2e'
    )
    ax2.text(
        cx, 0.22, label,
        transform=ax2.transAxes,
        ha='center', va='center',
        fontsize=7.5, color='#888888',
        fontfamily='sans-serif'
    )

# ── Title ──────────────────────────────────────────────────────────────────────
fig.suptitle(
    'Yearly SWE Average vs. Colorado Average Temperature  (1980–2025)',
    fontsize=13, fontweight='bold', color='#1a1a2e', y=0.95
)

# ── Save ───────────────────────────────────────────────────────────────────────
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print(f"\nPNG saved to: {OUTPUT_PATH}")

# ── Auto-download in Colab ─────────────────────────────────────────────────────
if IN_COLAB and os.path.exists(OUTPUT_PATH):
    files.download(OUTPUT_PATH)
    print("Downloading file...")

Mounted at /content/drive
── Regression Results ────────────────────────────────
  Pearson r  : -0.637076
  R²         : 0.405866
  P-value    : 1.932497e-06
  Slope      : -1.5681
  Intercept  : 17.3797
  Std Error  : 0.286019
  N          : 46
──────────────────────────────────────────────────────

PNG saved to: /content/swe_temp_regression.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>